In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [2]:
#load dataset
data = pd.read_csv("Churn Prediction.csv")

In [3]:
data.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data.shape

(10000, 12)

In [5]:
data['country'].value_counts()

country
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [6]:
data.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [7]:
#preprocess the data
data = data.drop(['customer_id'],axis=1)

In [8]:
lable_encoder_gender = LabelEncoder()
data['gender']=lable_encoder_gender.fit_transform(data['gender'])
data

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [9]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder()
country_encoder = onehot_encoder.fit_transform(data[['country']])
country_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [10]:
country_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [11]:
onehot_encoder.get_feature_names_out(['country'])

array(['country_France', 'country_Germany', 'country_Spain'], dtype=object)

In [12]:
country_encoder_df = pd.DataFrame(country_encoder.toarray(),columns=onehot_encoder.get_feature_names_out(['country']))
country_encoder_df

,country_France,country_Germany,country_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [13]:
data = pd.concat([data.drop('country',axis=1),country_encoder_df],axis=1)
data.head()

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [ ]:
#save the encoder files
with open('lable_encoder_gender.pkl','wb') as file:
    pickle.dump(lable_encoder_gender,file)

with open('onehot_encoder.pkl','wb') as file:
    pickle.dump(onehot_encoder,file)


In [16]:
X = data.drop('churn',axis=1)
y = data['churn']

In [17]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) 

In [19]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [26]:
(X_train.shape[1],)

(12,)

In [28]:
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [30]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [31]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

In [32]:
model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])

In [53]:
log_dir = "logs/fit/"+datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
tesorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [54]:
early_stopping_callback= EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [55]:
history = model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tesorboard_callback,early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 2s 6ms/step - loss: 0.3552 - accuracy: 0.8561 - val_loss: 0.3642 - val_accuracy: 0.8535
Epoch 2/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3487 - accuracy: 0.8554 - val_loss: 0.3434 - val_accuracy: 0.8625
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3438 - accuracy: 0.8605 - val_loss: 0.3379 - val_accuracy: 0.8590
Epoch 4/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3384 - accuracy: 0.8605 - val_loss: 0.3614 - val_accuracy: 0.8620
Epoch 5/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3357 - accuracy: 0.8620 - val_loss: 0.3419 - val_accuracy: 0.8605
Epoch 6/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3351 - accuracy: 0.8620 - val_loss: 0.3428 - val_accuracy: 0.8590
Epoch 7/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3317 - accuracy: 0.8625 - val_loss: 0.3485 - val_accuracy: 0.8550

In [43]:
model.save("model.h5")

c:\ANN PROJECT\.venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [51]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [56]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 1944), started 0:06:36 ago. (Use '!kill 1944' to kill it.)